<a href="https://colab.research.google.com/github/fjnarvaezo1-dotcom/Proyecto-de-Grado-UNAD---Chat-BOT-IA-integrado/blob/main/Proyecto_de_Grado_Chat_mesa_de_servicio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Proyecto de Grado Francisco Narvaez MOdelo de LLM - MESA DE SERVICIOS PREGUNTAS FRECUENTES
# Cedula  :  1023915471
# Especializacion de ciencia de Datos y Analitica

## - In this step, I install the libraries needed for this project; in this guide, I want to explain how to connect Notebook to Google Collab.

In [1]:

import os
from google.colab import userdata
from google import genai

try:
    import google.genai
    print("google-genai imported successfully!")
    # Display the package version to verify it is indeed installed
    import pkg_resources
    version = pkg_resources.get_distribution("google-genai").version
    print(f"Installed version: {version}")
except ImportError as e:
    print(f"Import failed: {e}")

google-genai imported successfully!
Installed version: 2.12.1


/tmp/ipykernel_497/1058287523.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
pip install -q -U google-genai chromadb pypdf gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [26]:
from google.colab import userdata
try:
    api_key = userdata.get('RAG')
    client = genai.Client(api_key=api_key)
    print("Cliente Gemini inicializado correctamente con Colab Secrets.")
except userdata.SecretNotFoundError:
    print("Error: Please make sure you have added a secret with the name 'GEMINI_API_KEY' in the Secrets tab (key icon) on the left.")


Cliente Gemini inicializado correctamente con Colab Secrets.


In [27]:
import getpass
import time
from google import genai
from google.genai.errors import ClientError, APIError

api_key = api_key
if not api_key.startswith("AIzaSy"):
    print("¡Atención! Las claves de Google AI Studio normalmente comienzan con 'AIzaSy'.")

try:
    client = genai.Client(api_key=api_key)

    # Implementamos un bucle de reintento robusto en caso de congestión (Error 503)
    max_retries = 5
    delay = 2
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents="Hola, confirma con un 'OK' si recibes este mensaje."
            )
            print(f"\nConexión exitosa: {response.text.strip()}")
            break
        except Exception as e:
            # Si el modelo está ocupado (503) o hay límite de cuota (429), reintentamos
            if "503" in str(e) or "UNAVAILABLE" in str(e) or "429" in str(e):
                if attempt < max_retries - 1:
                    print(f"Servidor ocupado (Intento {attempt + 1}/{max_retries}). Reintentando en {delay} segundos...")
                    time.sleep(delay)
                    delay *= 2  # Incremento exponencial del tiempo de espera
                else:
                    raise e
            else:
                raise e

except ClientError as e:
    if "API_KEY_INVALID" in str(e):
        print("\nError: La clave API ingresada no es válida. Por favor, genera una nueva clave en Google AI Studio (https://aistudio.google.com/) e ingresa una clave que comience con 'AIzaSy'.")
    else:
        print(f"\nError de API del Cliente: {e}")
except Exception as e:
    print(f"\nNo se pudo establecer la conexión: {e}")

¡Atención! Las claves de Google AI Studio normalmente comienzan con 'AIzaSy'.

Conexión exitosa: OK


## In this step , now I configurate form at i can upload document - ajustes de modelos de RAG para analisis de manuales e informacion con IA

In [28]:
os.makedirs("documentos_telefonia", exist_ok=True)

In [29]:
import os
import glob
import pypdf
from google.colab import drive

drive.mount('/content/drive')
carpeta_drive = "/content/drive/MyDrive/PROYECTO DE GRADO"

pdfs = [
    os.path.join(carpeta_drive, f)
    for f in os.listdir(carpeta_drive)
    if f.lower().endswith(".pdf")
]

print(f"PDFs encontrados ({len(pdfs)}):", [os.path.basename(p) for p in pdfs])

def extraer_chunks_pdf(ruta_pdf, chunk_size=700, chunk_overlap=100):
    chunks = []
    nombre_archivo = os.path.basename(ruta_pdf)
    lector = pypdf.PdfReader(ruta_pdf)

    for num_pag, pagina in enumerate(lector.pages):
        texto = (pagina.extract_text() or "").strip()
        if not texto:
            continue

        inicio = 0
        while inicio < len(texto):
            fin = inicio + chunk_size
            fragmento = texto[inicio:fin].strip()
            if fragmento:
                chunks.append({
                    "id": f"{nombre_archivo}_p{num_pag + 1}_{inicio}",
                    "texto": fragmento,
                    "fuente": nombre_archivo,
                    "pagina": num_pag + 1
                })
            inicio += (chunk_size - chunk_overlap)

    return chunks

todos_los_chunks = []
for ruta_pdf in pdfs:
    chunks_pdf = extraer_chunks_pdf(ruta_pdf)
    todos_los_chunks.extend(chunks_pdf)
    if len(chunks_pdf) == 0:
        print(f"Advertencia: '{os.path.basename(ruta_pdf)}' no devolvió texto (puede ser un PDF escaneado como imagen).")
    else:
        print(f"-> {os.path.basename(ruta_pdf)}: {len(chunks_pdf)} fragmentos extraídos.")

print(f"\nTotal de fragmentos listos para indexar: {len(todos_los_chunks)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PDFs encontrados (4): ['syslog-viewer-users-guide.pdf', 'Avaya_SBC_Working_with_Multi_Tenancy.pdf', 'Avaya_SBCE_Overview_And_Specification.pdf', 'mediant-software-sbc-users-manual-ver-74.pdf']
-> syslog-viewer-users-guide.pdf: 170 fragmentos extraídos.
-> Avaya_SBC_Working_with_Multi_Tenancy.pdf: 154 fragmentos extraídos.
-> Avaya_SBCE_Overview_And_Specification.pdf: 371 fragmentos extraídos.
-> mediant-software-sbc-users-manual-ver-74.pdf: 5820 fragmentos extraídos.

Total de fragmentos listos para indexar: 6515


# Indexar en ChromaDB (guardando la base en tu Drive)

In [30]:
!pip install -q sentence-transformers

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

ruta_chroma = os.path.join(carpeta_drive, "chroma_db_local")
chroma_client = chromadb.PersistentClient(path=ruta_chroma)

ef_local = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

try:
    chroma_client.delete_collection("telefonia_rag_local")
except Exception:
    pass

coleccion = chroma_client.create_collection(
    name="telefonia_rag_local",
    embedding_function=ef_local,
    metadata={"hnsw:space": "cosine"}
)

batch_size = 200
for i in range(0, len(todos_los_chunks), batch_size):
    lote = todos_los_chunks[i:i + batch_size]
    coleccion.add(
        ids=[item["id"] for item in lote],
        documents=[item["texto"] for item in lote],
        metadatas=[{"fuente": item["fuente"], "pagina": item["pagina"]} for item in lote]
    )
    print(f"Indexados {min(i + batch_size, len(todos_los_chunks))}/{len(todos_los_chunks)}...")

print(f"\n¡Listo! Total en ChromaDB: {coleccion.count()}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexados 200/6515...
Indexados 400/6515...


# Entrenamiento de IA y funcionalidades para chat web

In [ ]:
from google.genai import types

MODELO_EMBEDDINGS = "gemini-embedding-001"
MODELO_LLM = "gemini-3.8-flash"

def buscar_contexto(pregunta, n_results=4):
    # Convertir la pregunta del usuario a vector usando el mismo modelo
    emb_pregunta = client.models.embed_content(
        model=MODELO_EMBEDDINGS,
        contents=pregunta
    ).embeddings[0].values

    resultados = coleccion.query(
        query_embeddings=[emb_pregunta],
        n_results=n_results
    )

    if not resultados["documents"] or not resultados["documents"][0]:
        return "No se encontró información relacionada en los documentos."

    fragmentos = []
    for doc, meta in zip(resultados["documents"][0], resultados["metadatas"][0]):
        fragmentos.append(f"[Fuente: {meta['fuente']} | Pág. {meta['pagina']}]\n{doc}")

    return "\n\n---\n\n".join(fragmentos)


def responder_rag(mensaje, historial):
    try:
        contexto = buscar_contexto(mensaje, n_results=4)

        instruccion_sistema = """Eres un asistente experto en telefonía y telecomunicaciones.
Responde a la consulta del usuario de forma clara, amable y precisa basándote ÚNICAMENTE en el contexto proporcionado de los documentos.
- Cita siempre el nombre del documento y la página de donde tomaste la información.
- Si la respuesta no aparece en el contexto, dilo con honestidad sin inventar datos."""

        prompt_usuario = f"""Contexto recuperado de los documentos:
{contexto}

Pregunta del usuario:
{mensaje}"""

        respuesta = client.models.generate_content(
            model=MODELO_LLM,
            contents=prompt_usuario,
            config=types.GenerateContentConfig(
                system_instruction=instruccion_sistema,
                temperature=0.2
            )
        )
        return respuesta.text

    except Exception as e:
        return f"Ocurrió un error al consultar el sistema: {str(e)}"

# Chat interactivo

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=responder_rag,
    title="Asistente Virtual de Telefonía (RAG + Gemini)",
    description="Haz preguntas sobre los documentos de telefonía cargados en el sistema.",
    examples=[
        "¿De qué tratan los documentos cargados?",
        "¿Cuáles son los requisitos o parámetros técnicos principales?",
        "¿Qué procedimientos de soporte o configuración aparecen en los manuales?"
    ]
)

demo.launch(share=True, debug=True)
